In [3]:
spark

In [4]:
from pyspark.sql import SparkSession
spark=SparkSession.builder\
.appName("ReduceByKey And GroupByKey")\
.master("yarn")\
.getOrCreate()

25/12/18 18:27:45 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [5]:
!hadoop fs -ls /tmp/

Found 6 items
drwxr-xr-x   - root hadoop          0 2025-12-18 09:52 /tmp/1MB
drwxr-xr-x   - root hadoop          0 2025-12-05 09:56 /tmp/active_customers.csv
drwxrwxrwt   - hdfs hadoop          0 2025-10-09 05:57 /tmp/hadoop-yarn
drwx-wx-wx   - hive hadoop          0 2025-10-09 05:57 /tmp/hive
-rw-r--r--   2 root hadoop         72 2025-12-03 19:11 /tmp/input_spark.txt
-rw-r--r--   2 root hadoop     863301 2025-12-18 09:53 /tmp/orders.csv


In [17]:
hdfs_path='/tmp/orders.csv'
rdd = spark.sparkContext.textFile(hdfs_path)

In [18]:
!hadoop fs -head /tmp/orders.csv

order_id,customer_id,order_date,total_amount,status
0,3692,2024-09-03,547.7160076008001,Shipped
1,11055,2024-08-10,577.8942599188381,Pending
2,6963,2024-08-22,484.2085562764487,Pending
3,13268,2024-09-01,366.3286882431848,Cancelled
4,1131,2024-08-09,896.9588380686909,Pending
5,15211,2024-05-03,486.30584827618145,Shipped
6,3209,2024-08-24,800.9795667933956,Shipped
7,15964,2024-11-24,641.3521833737843,Cancelled
8,9697,2024-06-18,789.1864475667901,Delivered
9,8917,2024-09-01,878.8901928499223,Shipped
10,68,2024-02-01,250.90701173471632,Pending
11,2826,2024-10-22,830.3870224465686,Pending
12,436,2024-03-16,723.8916744984356,Pending
13,17191,2024-07-22,565.4621691131047,Cancelled
14,16938,2024-03-31,549.5139113791525,Pending
15,13254,2024-10-28,318.3055446959585,Delivered
16,8107,2024-01-26,545.2292782609134,Cancelled
17,14662,2024-04-16,390.50606498059494,Cancelled
18,17007,2024-09-26,889.9218191968122,Delivered
19,15730,2024-07-24,586.2554183766008,Pending
20,13743,2024-04-07,306.03955079

In [20]:
header = rdd.first()

In [21]:
rdd_no_header = rdd.filter(lambda row:row!=header).map(lambda row:row.split(','))

In [22]:
rdd_no_header.first()

['0', '3692', '2024-09-03', '547.7160076008001', 'Shipped']

In [23]:
reduced_by_rdd = rdd_no_header.map(lambda row:(row[1],1)).reduceByKey(lambda x,y:x+y)

In [24]:
reduced_by_rdd.collect()

[('11055', 2),
 ('13268', 2),
 ('15211', 2),
 ('3209', 1),
 ('8917', 1),
 ('68', 1),
 ('2826', 4),
 ('17191', 2),
 ('16938', 2),
 ('8107', 4),
 ('14662', 2),
 ('17007', 1),
 ('15730', 2),
 ('13743', 1),
 ('6990', 4),
 ('3516', 1),
 ('11465', 3),
 ('8476', 1),
 ('17005', 4),
 ('430', 2),
 ('3922', 1),
 ('1381', 3),
 ('3671', 1),
 ('13185', 2),
 ('9505', 3),
 ('2516', 1),
 ('17245', 1),
 ('12051', 1),
 ('1203', 1),
 ('8787', 2),
 ('15318', 1),
 ('12675', 1),
 ('7256', 4),
 ('6910', 4),
 ('2402', 1),
 ('14825', 1),
 ('11514', 1),
 ('3604', 3),
 ('10902', 2),
 ('13984', 1),
 ('3040', 1),
 ('2286', 1),
 ('17442', 1),
 ('638', 2),
 ('356', 3),
 ('7253', 3),
 ('6617', 1),
 ('296', 2),
 ('16364', 3),
 ('12461', 1),
 ('5786', 3),
 ('2202', 3),
 ('3747', 3),
 ('9990', 1),
 ('13165', 1),
 ('557', 2),
 ('10797', 2),
 ('16538', 1),
 ('12268', 2),
 ('14844', 2),
 ('7179', 2),
 ('187', 1),
 ('10879', 1),
 ('13942', 1),
 ('14713', 4),
 ('3017', 4),
 ('17571', 2),
 ('11448', 2),
 ('10498', 2),
 ('15251

In [25]:
grouped_by_rdd  = rdd_no_header.map(lambda row:(row[2],1)).groupByKey()

In [26]:
grouped_by_rdd.collect()

[('2024-09-03', <pyspark.resultiterable.ResultIterable at 0x7fd3daad9b10>),
 ('2024-08-22', <pyspark.resultiterable.ResultIterable at 0x7fd3da59aad0>),
 ('2024-11-24', <pyspark.resultiterable.ResultIterable at 0x7fd3da59aa50>),
 ('2024-06-18', <pyspark.resultiterable.ResultIterable at 0x7fd3da59a710>),
 ('2024-10-22', <pyspark.resultiterable.ResultIterable at 0x7fd3da59a810>),
 ('2024-03-16', <pyspark.resultiterable.ResultIterable at 0x7fd3da599b90>),
 ('2024-03-31', <pyspark.resultiterable.ResultIterable at 0x7fd3da59a650>),
 ('2024-06-11', <pyspark.resultiterable.ResultIterable at 0x7fd3da92a550>),
 ('2024-01-06', <pyspark.resultiterable.ResultIterable at 0x7fd3da59a610>),
 ('2024-04-24', <pyspark.resultiterable.ResultIterable at 0x7fd3da59b210>),
 ('2024-08-26', <pyspark.resultiterable.ResultIterable at 0x7fd3da59b290>),
 ('2024-04-04', <pyspark.resultiterable.ResultIterable at 0x7fd3da59b490>),
 ('2024-06-29', <pyspark.resultiterable.ResultIterable at 0x7fd3da59b510>),
 ('2024-07-2

In [27]:
grouped_by_result = grouped_by_rdd.map(lambda row:(row[0],len(row[1])))

In [28]:
grouped_by_result.collect()

[('2024-09-03', 41),
 ('2024-08-22', 51),
 ('2024-11-24', 39),
 ('2024-06-18', 57),
 ('2024-10-22', 51),
 ('2024-03-16', 55),
 ('2024-03-31', 52),
 ('2024-06-11', 40),
 ('2024-01-06', 48),
 ('2024-04-24', 55),
 ('2024-08-26', 58),
 ('2024-04-04', 51),
 ('2024-06-29', 54),
 ('2024-07-29', 45),
 ('2024-09-20', 50),
 ('2024-11-13', 44),
 ('2024-04-01', 38),
 ('2024-02-25', 43),
 ('2024-04-26', 59),
 ('2024-12-27', 67),
 ('2024-09-27', 50),
 ('2024-03-02', 52),
 ('2024-07-07', 68),
 ('2024-01-25', 43),
 ('2024-10-31', 40),
 ('2024-09-24', 52),
 ('2024-08-02', 56),
 ('2024-12-13', 54),
 ('2024-10-16', 52),
 ('2024-07-06', 46),
 ('2024-06-02', 51),
 ('2024-02-27', 51),
 ('2024-05-31', 59),
 ('2024-09-04', 45),
 ('2024-04-23', 44),
 ('2024-07-31', 35),
 ('2024-06-03', 38),
 ('2024-02-07', 36),
 ('2024-05-30', 45),
 ('2024-07-26', 48),
 ('2024-09-23', 61),
 ('2024-12-03', 55),
 ('2024-12-25', 49),
 ('2024-07-15', 54),
 ('2024-10-12', 42),
 ('2024-06-07', 54),
 ('2024-12-28', 47),
 ('2024-12-16